In [ ]:
import torch
from src.models.model import GalaxyClassifier_ResNet
from pytorch_grad_cam import GradCAM
from PIL import Image
import io
import base64
import numpy as np
import torch
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from torchvision import transforms
import mlflow
import random
 

In [2]:
def build_gradcam(model: torch.nn.Module) -> GradCAM:
    target_layer = [model.group5.units[-1].conv2]
    return GradCAM(model=model, target_layers=target_layer)

In [ ]:
model = mlflow.pytorch.load_model()

In [ ]:

def run_gradcam(
    cam: GradCAM,tensor: torch.Tensor,
    orig_img: Image.Image) -> str:
    

    """Return base64-encoded PNG heatmap overlay for a single image tensor."""

    grayscale_cam = cam(input_tensor=tensor.unsqueeze(0), targets=None)[0]
    img_rgb = np.array(orig_img.resize((128, 128))) / 255.0
    heatmap = show_cam_on_image(img_rgb.astype(np.float32), grayscale_cam, use_rgb=True)
    buf = io.BytesIO()
    Image.fromarray(heatmap).save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode()

In [ ]:
raw_transforms = transforms.Compose([
    transforms.CenterCrop(random.randint(170, 240)),  
    transforms.Resize(160),                            
    transforms.RandomCrop(128),                        
    transforms.RandomRotation(degrees = [90, 180]),
    transforms.RandomRotation(degrees = [0,270]),      
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.1376, 0.1276, 0.1187],
                         std= [0.1728, 0.1586, 0.1488])    
])


In [15]:
model = GalaxyClassifier_ResNet().eval()
gradcam = build_gradcam(model)


